In [1]:
%load_ext autoreload
%autoreload 2
import sys, time
sys.path.insert(0, "..")
from pathlib import Path
import numpy as np, pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report, confusion_matrix
from src.experiments import log_experiment

DATA, MODELS = Path("../data/processed"), Path("../models")
NUM_COLS = ["vader", "n_exclaim", "caps_ratio", "escalation_kw", "risk_kw", "calm_kw"]

sp = pd.read_parquet(DATA / "silver_priority.parquet")
sp[NUM_COLS] = sp[NUM_COLS].astype(float)
sp["n_exclaim"] = sp["n_exclaim"].clip(upper=5)

train, temp = train_test_split(sp, test_size=0.2, stratify=sp["priority"], random_state=42)
val, test = train_test_split(temp, test_size=0.5, stratify=temp["priority"], random_state=42)
pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])[
    ["tweet_id", "split"]].to_parquet(DATA / "ps2_splits.parquet", index=False)
print(f"train {len(train):,} | val {len(val):,} | test {len(test):,}")

train 44,384 | val 5,548 | test 5,548


In [2]:
def p_scores(y, p):
    return {"macro_f1": f1_score(y, p, average="macro"),
            "high_recall": recall_score(y, p, labels=["High"], average="macro"),
            "high_precision": precision_score(y, p, labels=["High"], average="macro", zero_division=0)}

tfidf = lambda **kw: TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True, **kw)
lr = lambda: LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0)

models = {
    "LR text-only": Pipeline([
        ("feat", ColumnTransformer([("tfidf", tfidf(max_features=50_000), "text_clean")])),
        ("clf", lr())]),
    "LR text+signals": Pipeline([
        ("feat", ColumnTransformer([("tfidf", tfidf(max_features=50_000), "text_clean"),
                                    ("num", StandardScaler(), NUM_COLS)])),
        ("clf", lr())]),
    "RF text+signals": Pipeline([
        ("feat", ColumnTransformer([("tfidf", tfidf(max_features=20_000), "text_clean"),
                                    ("num", "passthrough", NUM_COLS)])),
        ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample",
                                       n_jobs=-1, random_state=42))]),
}

results = {}
for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(train, train["priority"])
    m = p_scores(val["priority"], pipe.predict(val))
    results[name] = (pipe, m)
    log_experiment("PS-2", name, {"eval": "silver_val"}, m)
    print(f"{name:16}: macro-F1 {m['macro_f1']:.4f} | High recall {m['high_recall']:.3f} "
          f"| High precision {m['high_precision']:.3f} | {time.time() - t0:.0f}s")

best_name = max(results, key=lambda k: results[k][1]["macro_f1"])
best = results[best_name][0]
pred = best.predict(test)
print(f"\nBest: {best_name} → silver TEST\n")
print(classification_report(test["priority"], pred, digits=3))
labels = ["High", "Medium", "Low"]
print(pd.DataFrame(confusion_matrix(test["priority"], pred, labels=labels),
                   index=[f"true_{l}" for l in labels], columns=labels))
joblib.dump(best, MODELS / "ps2_best_classical.joblib")

LR text-only    : macro-F1 0.8033 | High recall 0.694 | High precision 0.666 | 3s
LR text+signals : macro-F1 0.8954 | High recall 0.899 | High precision 0.828 | 3s
RF text+signals : macro-F1 0.8604 | High recall 0.745 | High precision 0.838 | 4s

Best: LR text+signals → silver TEST

              precision    recall  f1-score   support

        High      0.833     0.878     0.855       278
         Low      0.918     0.925     0.921      2623
      Medium      0.913     0.901     0.907      2647

    accuracy                          0.911      5548
   macro avg      0.888     0.901     0.894      5548
weighted avg      0.911     0.911     0.911      5548

             High  Medium   Low
true_High     244      32     2
true_Medium    47    2385   215
true_Low        2     195  2426


['../models/ps2_best_classical.joblib']

In [3]:
import json
from tensorflow import keras
from tensorflow.keras import layers
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

keras.utils.set_random_seed(42)
SEQ_LEN, DIM = 40, 100
w2v = Word2Vec.load(str(MODELS / "w2v_tickets.model"))
as_arr = lambda d: np.array(d["text_clean"].tolist(), dtype=object)

vec = layers.TextVectorization(max_tokens=20_000, output_sequence_length=SEQ_LEN, standardize=None)
vec.adapt(as_arr(train))
vocab = vec.get_vocabulary()
emb = np.random.normal(0, 0.1, (len(vocab), DIM)).astype("float32")
emb[0] = 0
known = [i for i, w in enumerate(vocab) if w in w2v.wv]
emb[known] = np.stack([w2v.wv[vocab[i]] for i in known])

scaler = StandardScaler().fit(train[NUM_COLS])
le = LabelEncoder().fit(train["priority"])
inputs = lambda d: {"text": vec(as_arr(d)).numpy(),
                    "signals": scaler.transform(d[NUM_COLS]).astype("float32")}
Xtr, Xva, Xte = inputs(train), inputs(val), inputs(test)
ytr = le.transform(train["priority"])
yva = le.transform(val["priority"])

text_in = keras.Input(shape=(SEQ_LEN,), dtype="int64", name="text")
sig_in = keras.Input(shape=(len(NUM_COLS),), name="signals")
x = layers.Embedding(len(vocab), DIM, embeddings_initializer=keras.initializers.Constant(emb),
                     mask_zero=True)(text_in)
x = layers.Bidirectional(layers.LSTM(64))(x)
x = layers.Dropout(0.3)(x)
x = layers.Concatenate()([x, sig_in])
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(le.classes_), activation="softmax")(x)
model = keras.Model([text_in, sig_in], out)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(len(le.classes_)), y=ytr)))
t0 = time.time()
model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=10, batch_size=128, class_weight=cw, verbose=2,
          callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)])

predict = lambda X: le.inverse_transform(model.predict(X, verbose=0).argmax(axis=1))
m_val, m_test = p_scores(val["priority"], predict(Xva)), p_scores(test["priority"], predict(Xte))
log_experiment("PS-2", "BiLSTM+signals(W2V-init)", {"units": 64, "seq_len": SEQ_LEN, "eval": "silver_val"}, m_val)
print(f"\nBiLSTM+signals: val macro-F1 {m_val['macro_f1']:.4f} | High recall {m_val['high_recall']:.3f} || "
      f"test macro-F1 {m_test['macro_f1']:.4f} | High recall {m_test['high_recall']:.3f} | {time.time() - t0:.0f}s")

model.save(MODELS / "ps2_bilstm.keras")
joblib.dump({"scaler": scaler, "label_encoder": le, "num_cols": NUM_COLS}, MODELS / "ps2_bilstm_aux.joblib")
(MODELS / "ps2_vocab.json").write_text(json.dumps(vocab))

Epoch 1/10
347/347 - 15s - 43ms/step - accuracy: 0.8172 - loss: 0.4096 - val_accuracy: 0.8695 - val_loss: 0.3235
Epoch 2/10
347/347 - 13s - 38ms/step - accuracy: 0.8968 - loss: 0.2187 - val_accuracy: 0.8882 - val_loss: 0.2706
Epoch 3/10
347/347 - 15s - 42ms/step - accuracy: 0.9242 - loss: 0.1654 - val_accuracy: 0.8740 - val_loss: 0.3092
Epoch 4/10
347/347 - 14s - 41ms/step - accuracy: 0.9452 - loss: 0.1208 - val_accuracy: 0.8980 - val_loss: 0.2657
Epoch 5/10
347/347 - 15s - 43ms/step - accuracy: 0.9539 - loss: 0.0979 - val_accuracy: 0.9065 - val_loss: 0.2483
Epoch 6/10
347/347 - 15s - 43ms/step - accuracy: 0.9631 - loss: 0.0789 - val_accuracy: 0.9010 - val_loss: 0.2820
Epoch 7/10
347/347 - 15s - 42ms/step - accuracy: 0.9723 - loss: 0.0618 - val_accuracy: 0.8958 - val_loss: 0.3265

BiLSTM+signals: val macro-F1 0.8879 | High recall 0.910 || test macro-F1 0.8834 | High recall 0.899 | 103s


220648